[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/050.%20The%20New%20Product%20Forecasting%20Problem/P50-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 50. The New Product Forecasting Problem (Look-Alike Modeling)

## Tier 4: The AI/ML/RL Augmentation Method

This notebook demonstrates a neural-network forecasting approach with adversarial training. The goal is to predict six monthly demand values for a new product from product and market attributes, then examine how those predictions respond to bounded changes in the inputs.

### Goals, inputs, and outputs

- **Inputs:** Synthetic product attributes and a synthetic six-month sales history for 12 smartphone products. The model receives 21 product and market features per product; historical sales are used to construct the teaching targets, not as model inputs.
- **Method:** A feed-forward neural network trained by gradient descent on clean and adversarially perturbed feature vectors. FGSM and PGD generate the perturbations.
- **Outputs:** Training and validation loss histories, clean and adversarial test errors, perturbation-based prediction intervals, and plots that summarize these quantities.

### Scope and modeling assumptions

The data are generated for instruction, not measured from a real market. Each product's six-month target is created by multiplying its synthetic historical sales by a randomly generated growth factor. Consequently, the results do not establish real-world forecast accuracy or demonstrate that this model outperforms another tier. The example contains only 12 products, far fewer than would normally be needed to train and evaluate a neural forecasting model reliably.

Features are standardized using training data only. Adversarial perturbations are bounded in the standardized feature space using the \(L_\infty\) norm. This means the same numeric perturbation budget applies to each standardized feature, not the same physical change in every original unit. The model does not enforce nonnegative forecasts, account for product life cycles explicitly, or model market segments separately.

The intervals in this notebook are empirical prediction ranges obtained by adding small random perturbations to test inputs. They are not statistically calibrated confidence intervals and should not be interpreted as having guaranteed coverage.

### Method outline

1. Construct synthetic product and market features and six-month forecasting targets.
2. Split the examples into training, validation, and test sets; fit feature and target scalers on the training set.
3. Define a feed-forward neural network and its gradient-descent training rule.
4. Generate adversarial examples with FGSM and PGD, and train on clean and perturbed examples.
5. Evaluate clean and adversarial errors and summarize input-perturbation sensitivity.

### Neural network and training objective

A neural network maps a feature vector through layers of weighted sums and nonlinear activations. For layer \(\ell\), the calculation is

$$
 z^{(\ell)} = a^{(\ell-1)} W^{(\ell)} + b^{(\ell)},
 \qquad
 a^{(\ell)} = \operatorname{ReLU}(z^{(\ell)})
$$

for hidden layers; the output layer is linear so it can predict six continuous demand values. Here, \(a^{(0)}=x\) is the input vector, \(W^{(\ell)}\) and \(b^{(\ell)}\) are the layer's weights and biases, and ReLU is \(\max(0,z)\). The code implements these calculations in `SimpleNeuralNetwork.forward`.

For a batch of \(m\) examples with \(q=6\) forecast periods, the mean squared error is

$$
 L = \frac{1}{mq}\sum_{i=1}^{m}\sum_{t=1}^{q}
       (\hat y_{it}-y_{it})^2.
$$

Backpropagation applies the chain rule to find the derivatives of this loss with respect to each weight and bias. Gradient descent then updates each parameter \(\theta\) by

$$
 \theta \leftarrow \theta - \eta \nabla_{\theta}L,
$$

where \(\eta\) is the learning rate. The implementation computes these derivatives in `SimpleNeuralNetwork.backward`.

### Adversarial perturbations

An adversarial example is an input deliberately changed to increase prediction loss while keeping the change within a specified budget. This is a robustness probe, not a claim that the perturbed input is a likely market scenario. In this notebook, perturbations are constructed using the model and the known target values, so the attacks are white-box attacks.

Let \(x\) be an input, \(y\) its target, and \(\nabla_x L(f(x),y)\) the loss gradient with respect to the input. FGSM makes one signed step:

$$
 x_{\mathrm{FGSM}} = x + \epsilon\,\operatorname{sign}\!\left(\nabla_x L(f(x),y)\right).
$$

PGD makes repeated signed steps and projects the result back into the \(L_\infty\) neighborhood of the original input:

$$
 x^{(k+1)} = x + \operatorname{clip}\!\left(
 x^{(k)} + \alpha\,\operatorname{sign}(\nabla_x L(f(x^{(k)}),y))-x,
 -\epsilon,\epsilon\right).
$$

Here, \(f\) is the neural network, \(\epsilon\) is the perturbation budget, \(\alpha\) is the PGD step size, and \(k\) indexes the PGD steps. The code computes input gradients with central finite differences in `_compute_numerical_gradients`, then constructs the attacks in `fgsm_attack` and `pgd_attack`.

Training minimizes the same mean squared error over a combined batch containing clean examples and the adversarial examples generated by the configured attacks. This teaches the network to fit the targets while also exposing it to bounded input changes.

### Interpreting the evaluation

Mean absolute error (MAE) and root mean squared error (RMSE) summarize test-set forecast error in standardized target units. Adversarial MAE measures the error after the specified input attack. The robustness score reported here is a bounded relative degradation measure: it is 1 when adversarial MAE is no greater than clean MAE, decreases as adversarial MAE becomes larger, and is clipped at 0. It is a teaching metric, not a standard accuracy measure.

The feature-perturbation intervals show how predictions vary under small random changes to standardized test inputs. They do not capture uncertainty from limited training data, model choice, or future market variation.

### Strengths and limitations of this approach

A neural network can represent nonlinear relationships between inputs and targets, while adversarial training explicitly probes sensitivity to bounded feature changes. These properties may be useful when sufficient representative data and validation resources are available. They do not guarantee better forecasts than mathematical, heuristic, or metaheuristic approaches.

The approach also has important costs and limitations: it needs substantial representative training data, requires choices about architecture and hyperparameters, can be difficult to interpret, and may overfit. The small synthetic dataset here is suitable for illustrating code and method, but not for supporting business decisions or comparative performance claims.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class ProductFeatures:
    """Feature representation for one synthetic product."""

    product_id: int
    name: str
    basic_features: np.ndarray
    technical_features: np.ndarray
    market_features: np.ndarray
    temporal_features: np.ndarray
    historical_sales: np.ndarray
    similarity_score: float


@dataclass
class NeuralNetworkConfig:
    """Configuration for the feed-forward neural network."""

    input_dim: int
    hidden_layers: List[int]
    output_dim: int
    activation: str = "relu"
    learning_rate: float = 0.001
    batch_size: int = 32
    epochs: int = 200


@dataclass
class AdversarialConfig:
    """Configuration for FGSM and PGD input perturbations."""

    epsilon: float = 0.1
    attack_methods: Optional[List[str]] = None
    pgd_steps: int = 7
    alpha: float = 0.01

    def __post_init__(self):
        if self.attack_methods is None:
            self.attack_methods = ["FGSM", "PGD"]


@dataclass
class MLResult:
    """Container for machine-learning evaluation results."""

    model_performance: Dict[str, float]
    robustness_scores: Dict[str, float]
    training_history: Dict[str, List[float]]
    confidence_intervals: Dict[str, np.ndarray]
    computation_time: float
    model_params: int

In [ ]:
class SimpleNeuralNetwork:
    """A small neural network trained with backpropagation and gradient descent."""

    def __init__(self, config: NeuralNetworkConfig):
        self.config = config
        self.weights = []
        self.biases = []
        self._initialize_network()

    def _initialize_network(self):
        """Initialize layer parameters so all required matrix products are defined."""
        layer_sizes = (
            [self.config.input_dim]
            + self.config.hidden_layers
            + [self.config.output_dim]
        )

        for layer_index in range(len(layer_sizes) - 1):
            fan_in = layer_sizes[layer_index]
            fan_out = layer_sizes[layer_index + 1]
            weight_matrix = np.random.randn(fan_in, fan_out) * np.sqrt(2.0 / fan_in)
            bias_vector = np.zeros(fan_out)
            self.weights.append(weight_matrix)
            self.biases.append(bias_vector)

    @staticmethod
    def _relu(values):
        """Apply ReLU, the hidden-layer activation specified for this example."""
        return np.maximum(0.0, values)

    @staticmethod
    def _relu_derivative(values):
        """Return the ReLU derivative needed by the backpropagation chain rule."""
        return (values > 0.0).astype(float)

    def forward(self, X):
        """Map a batch of feature vectors to six continuous forecasts."""
        self.activations = [X]
        current = X

        for layer_index, (weights, biases) in enumerate(
            zip(self.weights, self.biases)
        ):
            current = np.dot(current, weights) + biases
            if layer_index < len(self.weights) - 1:
                current = self._relu(current)
            self.activations.append(current)

        return current

    def backward(self, X, y, learning_rate):
        """Update weights and biases using the gradient of batch mean squared error."""
        sample_count, output_count = y.shape
        predictions = self.forward(X)

        # The factor 2 and output count match the mean over all batch targets.
        output_gradient = 2.0 * (predictions - y) / (sample_count * output_count)
        gradients_w = [None] * len(self.weights)
        gradients_b = [None] * len(self.biases)

        gradients_w[-1] = np.dot(self.activations[-2].T, output_gradient)
        gradients_b[-1] = np.sum(output_gradient, axis=0)

        hidden_gradient = output_gradient
        for layer_index in range(len(self.weights) - 2, -1, -1):
            hidden_gradient = np.dot(
                hidden_gradient, self.weights[layer_index + 1].T
            ) * self._relu_derivative(self.activations[layer_index + 1])
            gradients_w[layer_index] = np.dot(
                self.activations[layer_index].T, hidden_gradient
            )
            gradients_b[layer_index] = np.sum(hidden_gradient, axis=0)

        for layer_index in range(len(self.weights)):
            self.weights[layer_index] -= learning_rate * gradients_w[layer_index]
            self.biases[layer_index] -= learning_rate * gradients_b[layer_index]

    def predict(self, X):
        """Return model forecasts without changing the learned parameters."""
        return self.forward(X)


class AdversarialTrainer:
    """Train and evaluate a neural network with bounded adversarial inputs."""

    def __init__(self, config: NeuralNetworkConfig, adv_config: AdversarialConfig):
        self.config = config
        self.adv_config = adv_config
        self.model = SimpleNeuralNetwork(config)

    def fgsm_attack(self, X, y, epsilon):
        """Create one signed-gradient perturbation within an L-infinity budget."""
        gradients = self._compute_numerical_gradients(X, y)
        return X + epsilon * np.sign(gradients)

    def pgd_attack(self, X, y, epsilon, alpha, steps):
        """Take repeated signed-gradient steps and project onto an L-infinity ball."""
        X_adv = X.copy()

        for _ in range(steps):
            gradients = self._compute_numerical_gradients(X_adv, y)
            X_adv = X_adv + alpha * np.sign(gradients)
            perturbation = np.clip(X_adv - X, -epsilon, epsilon)
            X_adv = X + perturbation

        return X_adv

    def _compute_numerical_gradients(self, X, y, h=1e-5):
        """Estimate each input derivative by central finite differences."""
        sample_count, feature_count = X.shape
        row_indices = np.repeat(np.arange(sample_count), feature_count)
        feature_indices = np.tile(np.arange(feature_count), sample_count)
        X_plus = X[row_indices].copy()
        X_minus = X[row_indices].copy()
        X_plus[np.arange(len(row_indices)), feature_indices] += h
        X_minus[np.arange(len(row_indices)), feature_indices] -= h

        predictions_plus = self.model.predict(X_plus).reshape(
            sample_count, feature_count, -1
        )
        predictions_minus = self.model.predict(X_minus).reshape(
            sample_count, feature_count, -1
        )
        targets = y[:, np.newaxis, :]
        loss_plus = np.mean((predictions_plus - targets) ** 2, axis=2)
        loss_minus = np.mean((predictions_minus - targets) ** 2, axis=2)

        return (loss_plus - loss_minus) / (2.0 * h)

    def train_with_adversarial(self, X_train, y_train, X_val, y_val):
        """Train on clean and configured adversarial examples, then record diagnostics."""
        training_history = {"loss": [], "val_loss": [], "robustness": []}
        start_time = time.time()

        for epoch in range(self.config.epochs):
            epoch_loss = 0.0
            n_batches = 0

            for start in range(0, len(X_train), self.config.batch_size):
                end = min(start + self.config.batch_size, len(X_train))
                X_batch = X_train[start:end]
                y_batch = y_train[start:end]
                feature_batches = [X_batch]
                target_batches = [y_batch]

                if "FGSM" in self.adv_config.attack_methods:
                    feature_batches.append(
                        self.fgsm_attack(X_batch, y_batch, self.adv_config.epsilon)
                    )
                    target_batches.append(y_batch)

                if "PGD" in self.adv_config.attack_methods:
                    feature_batches.append(
                        self.pgd_attack(
                            X_batch,
                            y_batch,
                            self.adv_config.epsilon,
                            self.adv_config.alpha,
                            self.adv_config.pgd_steps,
                        )
                    )
                    target_batches.append(y_batch)

                X_combined = np.vstack(feature_batches)
                y_combined = np.vstack(target_batches)
                self.model.backward(X_combined, y_combined, self.config.learning_rate)

                predictions = self.model.predict(X_combined)
                epoch_loss += np.mean((predictions - y_combined) ** 2)
                n_batches += 1

            val_predictions = self.model.predict(X_val)
            val_loss = np.mean((val_predictions - y_val) ** 2)
            robustness = self._calculate_robustness(X_val, y_val)
            training_history["loss"].append(epoch_loss / n_batches)
            training_history["val_loss"].append(val_loss)
            training_history["robustness"].append(robustness)

            if epoch % 20 == 0:
                print(
                    f"Epoch {epoch}: Loss={epoch_loss / n_batches:.4f}, "
                    f"Validation loss={val_loss:.4f}, Robustness={robustness:.3f}"
                )

        return training_history, time.time() - start_time

    @staticmethod
    def _robustness_score(clean_mae, adversarial_mae):
        """Measure relative MAE degradation, clipped to the interval from zero to one."""
        denominator = max(clean_mae, np.finfo(float).eps)
        degradation = max(0.0, adversarial_mae - clean_mae) / denominator
        return float(np.clip(1.0 - degradation, 0.0, 1.0))

    def _calculate_robustness(self, X, y):
        """Evaluate relative FGSM degradation on the supplied data."""
        clean_predictions = self.model.predict(X)
        clean_mae = mean_absolute_error(y, clean_predictions)

        if "FGSM" not in self.adv_config.attack_methods:
            return 1.0

        X_adv = self.fgsm_attack(X, y, self.adv_config.epsilon)
        adv_mae = mean_absolute_error(y, self.model.predict(X_adv))
        return self._robustness_score(clean_mae, adv_mae)

    def evaluate_comprehensive(self, X_test, y_test):
        """Measure clean and adversarial test errors and input-perturbation ranges."""
        results = {}
        clean_predictions = self.model.predict(X_test)
        clean_mae = mean_absolute_error(y_test, clean_predictions)
        results["clean_mae"] = clean_mae
        results["clean_rmse"] = float(
            np.sqrt(mean_squared_error(y_test, clean_predictions))
        )
        target_scale = max(float(np.mean(np.abs(y_test))), np.finfo(float).eps)
        results["clean_relative_mae_score"] = float(1.0 - clean_mae / target_scale)

        if "FGSM" in self.adv_config.attack_methods:
            X_adv_fgsm = self.fgsm_attack(
                X_test, y_test, self.adv_config.epsilon
            )
            fgsm_mae = mean_absolute_error(
                y_test, self.model.predict(X_adv_fgsm)
            )
            results["fgsm_mae"] = fgsm_mae
            results["fgsm_robustness"] = self._robustness_score(
                clean_mae, fgsm_mae
            )

        if "PGD" in self.adv_config.attack_methods:
            X_adv_pgd = self.pgd_attack(
                X_test,
                y_test,
                self.adv_config.epsilon,
                self.adv_config.alpha,
                self.adv_config.pgd_steps,
            )
            pgd_mae = mean_absolute_error(y_test, self.model.predict(X_adv_pgd))
            results["pgd_mae"] = pgd_mae
            results["pgd_robustness"] = self._robustness_score(
                clean_mae, pgd_mae
            )

        predictions_under_noise = []
        for _ in range(100):
            X_noisy = X_test + np.random.normal(0.0, 0.01, X_test.shape)
            predictions_under_noise.append(self.model.predict(X_noisy))

        predictions_under_noise = np.asarray(predictions_under_noise)
        results["confidence_intervals"] = {
            "lower": np.percentile(predictions_under_noise, 2.5, axis=0),
            "upper": np.percentile(predictions_under_noise, 97.5, axis=0),
            "mean": np.mean(predictions_under_noise, axis=0),
        }
        return results

In [ ]:
def create_ml_dataset():
    """Create 12 reproducible synthetic smartphone examples."""
    np.random.seed(42)
    products = []
    product_names = [
        "iPhone_Pro_2019", "Galaxy_Ultra_2020", "Pixel_Advanced_2020",
        "OnePlus_Elite_2021", "Xiaomi_Flagship_2021", "Oppo_Pro_2022",
        "Vivo_Premium_2022", "Realme_Ultra_2023", "Nothing_Phone_2023",
        "Sony_Xperia_2024", "Nokia_Flagship_2024", "Motorola_Edge_2025",
    ]

    for product_id, name in enumerate(product_names):
        basic_features = np.array([
            np.random.uniform(699, 1299),
            np.random.uniform(6.1, 6.7),
            np.random.uniform(4000, 5000),
            np.random.uniform(180, 220),
            np.random.uniform(128, 512),
        ])
        technical_features = np.array([
            np.random.uniform(70, 95),
            np.random.uniform(6, 12),
            np.random.uniform(48, 108),
            np.random.uniform(60, 120),
            np.random.uniform(65, 68),
            np.random.uniform(75, 90),
            np.random.uniform(70, 90),
            np.random.uniform(60, 85),
        ])
        market_features = np.array([
            np.random.uniform(60, 95),
            np.random.uniform(5, 25),
            np.random.uniform(0.6, 0.9),
            np.random.uniform(0.3, 0.8),
        ])
        temporal_features = np.array([
            np.random.uniform(1, 4),
            np.random.uniform(0.4, 0.8),
            np.random.uniform(0.6, 0.9),
        ])

        base_sales = np.random.uniform(50, 150)
        trend = np.random.uniform(-5, 5)
        seasonality = np.sin(np.linspace(0, 2 * np.pi, 6)) * 20
        noise = np.random.normal(0, 10, 6)
        historical_sales = np.maximum(
            base_sales + trend * np.arange(6) + seasonality + noise, 10
        )

        products.append(
            ProductFeatures(
                product_id=product_id,
                name=name,
                basic_features=basic_features,
                technical_features=technical_features,
                market_features=market_features,
                temporal_features=temporal_features,
                historical_sales=historical_sales,
                similarity_score=np.random.uniform(0.4, 0.9),
            )
        )

    return products


products = create_ml_dataset()
feature_count = (
    len(products[0].basic_features)
    + len(products[0].technical_features)
    + len(products[0].market_features)
    + len(products[0].temporal_features)
    + 1
)
print(f"Created a synthetic dataset with {len(products)} products.")
print(f"Each model input contains {feature_count} features.")
print("Each forecasting target contains 6 monthly values.")

In [ ]:
def prepare_training_data(products):
    """Build features and targets, split examples, and scale from training data."""
    X = []
    y = []

    for product in products:
        feature_vector = np.concatenate([
            product.basic_features,
            product.technical_features,
            product.market_features,
            product.temporal_features,
            [product.similarity_score],
        ])
        growth_factor = np.random.uniform(0.8, 1.3)
        forecast_target = product.historical_sales * growth_factor
        X.append(feature_vector)
        y.append(forecast_target)

    X = np.asarray(X)
    y = np.asarray(y)
    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
        X_train_raw, y_train_raw, test_size=0.2, random_state=42
    )

    # Fitting scalers only on training examples prevents information leakage.
    scaler_X = StandardScaler().fit(X_train_raw)
    scaler_y = StandardScaler().fit(y_train_raw)
    X_train = scaler_X.transform(X_train_raw)
    X_val = scaler_X.transform(X_val_raw)
    X_test = scaler_X.transform(X_test_raw)
    y_train = scaler_y.transform(y_train_raw)
    y_val = scaler_y.transform(y_val_raw)
    y_test = scaler_y.transform(y_test_raw)

    return (
        X_train, X_val, X_test, y_train, y_val, y_test, scaler_X, scaler_y
    )


(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test,
    scaler_X,
    scaler_y,
) = prepare_training_data(products)

print(f"Training data shape: {X_train.shape}")
print(f"Validation data shape: {X_val.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")

In [ ]:
def train_adversarial_model():
    """Configure, train, and evaluate the adversarial neural network."""
    nn_config = NeuralNetworkConfig(
        input_dim=X_train.shape[1],
        hidden_layers=[128, 64, 32],
        output_dim=6,
        learning_rate=0.001,
        batch_size=32,
        epochs=200,
    )
    adv_config = AdversarialConfig(
        epsilon=0.1,
        attack_methods=["FGSM", "PGD"],
        pgd_steps=7,
        alpha=0.01,
    )
    trainer = AdversarialTrainer(nn_config, adv_config)

    print("Starting adversarial training.")
    print(
        f"Network architecture: {nn_config.input_dim} -> "
        f"{nn_config.hidden_layers} -> {nn_config.output_dim}"
    )
    print(f"Attack methods: {adv_config.attack_methods}")
    print(f"Perturbation budget (epsilon): {adv_config.epsilon}")
    print(f"Training examples: {len(X_train)}")

    training_history, computation_time = trainer.train_with_adversarial(
        X_train, y_train, X_val, y_val
    )
    test_results = trainer.evaluate_comprehensive(X_test, y_test)
    total_params = sum(
        weights.size + biases.size
        for weights, biases in zip(trainer.model.weights, trainer.model.biases)
    )
    return trainer, training_history, test_results, computation_time, total_params


trainer, training_history, test_results, computation_time, total_params = (
    train_adversarial_model()
)
print(f"Training completed in {computation_time:.2f} seconds.")
print(f"Total model parameters: {total_params:,}")

In [ ]:
def create_adversarial_visualizations(training_history, test_results):
    """Plot training behavior, test errors, robustness, and input-perturbation ranges."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "Adversarial Training for Look-Alike Forecasting",
        fontsize=16,
        fontweight="bold",
    )

    epochs = np.arange(1, len(training_history["loss"]) + 1)
    ax1 = axes[0, 0]
    ax1.plot(epochs, training_history["loss"], label="Training loss")
    ax1.plot(epochs, training_history["val_loss"], label="Validation loss")
    ax1.set_title("Training and validation progress")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Mean squared error (standardized units)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    ax2.plot(epochs, training_history["robustness"], color="green")
    ax2.set_title("FGSM robustness during training")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Relative degradation score")
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    error_names = ["Clean", "FGSM", "PGD"]
    error_values = [test_results["clean_mae"]]
    if "fgsm_mae" in test_results:
        error_values.append(test_results["fgsm_mae"])
    else:
        error_names.remove("FGSM")
    if "pgd_mae" in test_results:
        error_values.append(test_results["pgd_mae"])
    else:
        error_names.remove("PGD")
    bars = ax3.bar(error_names, error_values, color=["steelblue", "darkorange", "seagreen"][:len(error_values)])
    ax3.set_title("Test mean absolute error by input condition")
    ax3.set_ylabel("MAE (standardized target units)")
    ax3.grid(True, alpha=0.3, axis="y")
    for bar, value in zip(bars, error_values):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.3f}",
            ha="center",
            va="bottom",
        )

    ax4 = axes[1, 1]
    intervals = test_results["confidence_intervals"]
    sample_count = min(5, len(intervals["mean"]))
    months = np.arange(1, intervals["mean"].shape[1] + 1)
    for sample_index in range(sample_count):
        mean = intervals["mean"][sample_index]
        lower = intervals["lower"][sample_index]
        upper = intervals["upper"][sample_index]
        ax4.plot(months, mean, marker="o", label=f"Test example {sample_index + 1}")
        ax4.fill_between(months, lower, upper, alpha=0.12)
    ax4.set_title("Prediction ranges under small input perturbations")
    ax4.set_xlabel("Forecast month")
    ax4.set_ylabel("Prediction (standardized target units)")
    ax4.legend(fontsize=8)
    ax4.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
    return fig


fig = create_adversarial_visualizations(training_history, test_results)

In [ ]:
def analyze_model_performance(test_results, scaler_y):
    """Report clean and adversarial errors and interpret input-sensitivity ranges."""
    print("TEST-SET PERFORMANCE ANALYSIS")
    print("=" * 40)
    print(f"Clean MAE: {test_results['clean_mae']:.4f} standardized units")
    print(f"Clean RMSE: {test_results['clean_rmse']:.4f} standardized units")
    print(
        "Clean relative MAE score: "
        f"{test_results['clean_relative_mae_score']:.4f}"
    )

    if "fgsm_mae" in test_results:
        print(f"FGSM MAE: {test_results['fgsm_mae']:.4f} standardized units")
        print(f"FGSM robustness score: {test_results['fgsm_robustness']:.4f}")
    if "pgd_mae" in test_results:
        print(f"PGD MAE: {test_results['pgd_mae']:.4f} standardized units")
        print(f"PGD robustness score: {test_results['pgd_robustness']:.4f}")

    intervals = test_results["confidence_intervals"]
    average_width = float(np.mean(intervals["upper"] - intervals["lower"]))
    print("\nInput-perturbation sensitivity")
    print(f"Mean empirical interval width: {average_width:.4f} standardized units")
    print("These ranges are not calibrated statistical confidence intervals.")

    # The scaler has one output column per forecast month, so invert the complete row.
    sample_mean = scaler_y.inverse_transform(intervals["mean"][:1])[0]
    sample_lower = scaler_y.inverse_transform(intervals["lower"][:1])[0]
    sample_upper = scaler_y.inverse_transform(intervals["upper"][:1])[0]
    print("\nExample test forecast under small input perturbations")
    print(f"Month 1 mean prediction: {sample_mean[0]:.1f} thousand units")
    print(
        "Month 1 empirical prediction range: "
        f"[{sample_lower[0]:.1f}, {sample_upper[0]:.1f}] thousand units"
    )
    return test_results


performance_analysis = analyze_model_performance(test_results, scaler_y)

In [ ]:
def compare_tier_characteristics():
    """Explain what this example can and cannot establish about the four tiers."""
    comparison = [
        {
            "Tier": "Tier 1: Mathematical",
            "Method characteristic": "Explicit mathematical formulation",
            "Evidence in this notebook": "No Tier 1 experiment is included",
        },
        {
            "Tier": "Tier 2: Heuristic",
            "Method characteristic": "Rule-based or constructive decisions",
            "Evidence in this notebook": "No Tier 2 experiment is included",
        },
        {
            "Tier": "Tier 3: Metaheuristic",
            "Method characteristic": "Search over candidate solutions",
            "Evidence in this notebook": "No Tier 3 experiment is included",
        },
        {
            "Tier": "Tier 4: AI/ML",
            "Method characteristic": "Learned nonlinear mapping with adversarial training",
            "Evidence in this notebook": "Synthetic demonstration; not a benchmark",
        },
    ]
    comparison_table = pd.DataFrame(comparison)
    print(comparison_table.to_string(index=False))
    print(
        "\nA quantitative ranking requires comparable data, evaluation rules, and "
        "measured results for each method. This notebook supplies none of those "
        "cross-tier benchmark results."
    )
    return comparison_table


# pandas is imported here to keep the comparison table close to its use.
import pandas as pd

tier_comparison = compare_tier_characteristics()

In [ ]:
def print_ml_summary(trainer, test_results, computation_time, total_params):
    """Summarize the configured model, evaluation, and limits of the demonstration."""
    print("\n" + "=" * 68)
    print("ADVERSARIAL NEURAL-NETWORK FORECASTING SUMMARY")
    print("=" * 68)

    print("\nModel architecture")
    print(f"Input dimension: {trainer.config.input_dim} features")
    print(f"Hidden layers: {trainer.config.hidden_layers}")
    print(f"Output dimension: {trainer.config.output_dim} monthly values")
    print(f"Activation: {trainer.config.activation}")
    print(f"Trainable parameters: {total_params:,}")

    print("\nAdversarial training configuration")
    print(f"Attack methods: {trainer.adv_config.attack_methods}")
    print(f"L-infinity perturbation budget: {trainer.adv_config.epsilon}")
    print(f"PGD steps: {trainer.adv_config.pgd_steps}")
    print(f"Training epochs: {trainer.config.epochs}")
    print(f"Batch size: {trainer.config.batch_size}")

    print("\nTest metrics")
    print(f"Clean MAE: {test_results['clean_mae']:.4f} standardized units")
    print(f"Clean RMSE: {test_results['clean_rmse']:.4f} standardized units")
    if "fgsm_mae" in test_results:
        print(f"FGSM MAE: {test_results['fgsm_mae']:.4f} standardized units")
        print(f"FGSM robustness score: {test_results['fgsm_robustness']:.4f}")
    if "pgd_mae" in test_results:
        print(f"PGD MAE: {test_results['pgd_mae']:.4f} standardized units")
        print(f"PGD robustness score: {test_results['pgd_robustness']:.4f}")
    print(f"Training time: {computation_time:.2f} seconds")

    print("\nInterpretation and limitations")
    print("The dataset is synthetic and contains only 12 product examples.")
    print("Perturbation ranges describe input sensitivity, not calibrated coverage.")
    print("The example does not establish comparative or real-market performance.")
    print("A practical study needs representative historical data and validation.")
    print("=" * 68)


print_ml_summary(trainer, test_results, computation_time, total_params)